<a href="https://www.kaggle.com/code/shrikantlala/rezno-ai-backend?scriptVersionId=370412094" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
!pip install -U langchain-google-genai
!pip install arxiv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.1/588.1 kB 4.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 15.7 MB/s  0:00:00
  Attempting uninstall: google-auth
    Found existing installation: google-auth 2.49.0
    Uninstalling google-auth-2.49.0:
      Successfully uninstalled google-auth-2.49.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.0
    Uninstalling langchain-core-1.7.0:
      Successfully uninstalled langchain-core-1.7.0
  Attempting uninstall: google-genai
    Found existing installation: google-genai 2.12.1
    Uninstalling google-genai-2.12.1:
      Successfully uninstalled google-genai-2.12.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [langchain-google-genai]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

from IPython.display import display, Markdown
from dotenv import load_dotenv
import os

from pydantic import BaseModel, Field

import arxiv

In [3]:
load_dotenv()

False

In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("GOOGLE_API_KEY")


In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.7,
    google_api_key=secret_value_0
)

> alright let's define types first
---
1. arxiv_papers



In [6]:
# client = arxiv.Client
# client

def arxiv_search(query: str, max_results=10):

    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    client = arxiv.Client()

    papers = []

    for paper in client.results(search):

        papers.appenf({
            "id": paper.entry_id,
            "title": paper.title,
            "abstract": paper.summary,
            "authors": [a.name for a in paper.authors],
            "published": paper.published,
            "categories": paper.categories,
            "pdf_url": paper.pdf_url
        })

    return papers

In [7]:
class ResearchQueries(BaseModel):
    queries: list[str] = Field(
        description="4 to 10 relevant research search queries"
    )

In [8]:
def subquery_generater(user_input : str, sub_queries=3):

    prompt = ChatPromptTemplate.from_messages([
        ("system","You are an expert researcher and you know How a given concept/topic/theory of any given topic is connected to another. and you have ability to connect based on relevance and likelihood of occurence of them together"),
        ("human","given a user query '{user_input}', give me {sub_queries} no. of sub_queries which are connected with the main query in such way.")
    ])

    structured_llm = llm.with_structured_output(
        ResearchQueries,
        method="json_schema"
    )

    chain = prompt | structured_llm

    return chain.invoke({
        "user_query" : user_query,
        "sub_queries" : sub_queries
    })